# 03 — Final Feature Selection: Low Variance → ANOVA F-score → RFE

> Clean senior-review version. This notebook contains only the final, logically connected pipeline. 
> Exploratory/probing/failed scripts are intentionally excluded from the main workflow.

## Objective
Use the complete three-step feature-selection pipeline and save the **RFE-selected final feature subset** for model construction. This corrects the inconsistency where only F-score features were used later.

In [ ]:
# !pip install -q pandas numpy scikit-learn matplotlib joblib

In [ ]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif, RFE
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

PROJECT_DIR = Path.cwd()
MATRIX_FILE = PROJECT_DIR / "outputs" / "recA_chembl" / "fingerprints" / "recA_modeling_matrix.csv"
FS_DIR = PROJECT_DIR / "outputs" / "recA_chembl" / "feature_selection"
FS_DIR.mkdir(parents=True, exist_ok=True)

META_COLUMNS = ["Name", "SMILES", "EC50_nM", "pEC50", "bioactivity_class"]
LOW_VARIANCE_THRESHOLD = 0.01
TOP_K_FSCORE = 300
FINAL_RFE_FEATURES = 100
RANDOM_STATE = 42

## 1. Load modeling matrix

In [ ]:
df = pd.read_csv(MATRIX_FILE)
missing = [c for c in META_COLUMNS if c not in df.columns]
if missing:
    raise ValueError(f"Missing metadata columns: {missing}")

feature_cols = [c for c in df.columns if c not in META_COLUMNS]
X = df[feature_cols].apply(pd.to_numeric, errors="coerce").fillna(0)
y = df["bioactivity_class"].astype(int)

print("Matrix:", df.shape)
print("Features:", X.shape[1])
print(y.value_counts())

## 2. Step 1 — Low-variance filtering

In [ ]:
variance_selector = VarianceThreshold(threshold=LOW_VARIANCE_THRESHOLD)
X_var = variance_selector.fit_transform(X)
var_features = X.columns[variance_selector.get_support()].tolist()
X_var_df = pd.DataFrame(X_var, columns=var_features, index=X.index)

pd.DataFrame({"feature": var_features}).to_csv(FS_DIR / "01_low_variance_features.csv", index=False)
joblib.dump(variance_selector, FS_DIR / "variance_selector.joblib")

print("Features after low-variance filtering:", len(var_features))

## 3. Step 2 — ANOVA F-score ranking

In [ ]:
k = min(TOP_K_FSCORE, X_var_df.shape[1])
f_selector = SelectKBest(score_func=f_classif, k=k)
X_f = f_selector.fit_transform(X_var_df, y)

f_scores = pd.DataFrame({
    "feature": X_var_df.columns,
    "f_score": f_selector.scores_,
    "p_value": f_selector.pvalues_,
}).sort_values("f_score", ascending=False)

fscore_features = f_scores.head(k)["feature"].tolist()
X_f_df = X_var_df[fscore_features].copy()

f_scores.to_csv(FS_DIR / "02_anova_fscore_ranking.csv", index=False)
pd.DataFrame({"feature": fscore_features}).to_csv(FS_DIR / "02_fscore_top_features.csv", index=False)
joblib.dump(f_selector, FS_DIR / "fscore_selector.joblib")

print("Top F-score features passed to RFE:", len(fscore_features))
f_scores.head(10)

## 4. Step 3 — RFE final feature selection

In [ ]:
rfe_n = min(FINAL_RFE_FEATURES, X_f_df.shape[1])

base_estimator = RandomForestClassifier(
    n_estimators=300,
    random_state=RANDOM_STATE,
    class_weight="balanced",
    n_jobs=-1,
)

rfe = RFE(
    estimator=base_estimator,
    n_features_to_select=rfe_n,
    step=0.10,
)

rfe.fit(X_f_df, y)

rfe_ranking = pd.DataFrame({
    "feature": X_f_df.columns,
    "rfe_selected": rfe.support_,
    "rfe_rank": rfe.ranking_,
}).sort_values(["rfe_rank", "feature"])

final_features = rfe_ranking.loc[rfe_ranking["rfe_selected"], "feature"].tolist()

rfe_ranking.to_csv(FS_DIR / "03_rfe_feature_ranking.csv", index=False)
pd.DataFrame({"feature": final_features}).to_csv(FS_DIR / "final_selected_features_RFE.csv", index=False)
joblib.dump(rfe, FS_DIR / "rfe_selector.joblib")

print("Final RFE-selected features:", len(final_features))
rfe_ranking.head(15)

## 5. Export final modeling dataset

In [ ]:
final_dataset = pd.concat(
    [df[META_COLUMNS].reset_index(drop=True), X[final_features].reset_index(drop=True)],
    axis=1,
)

FINAL_DATASET_FILE = FS_DIR / "recA_RFE_final_modeling_dataset.csv"
final_dataset.to_csv(FINAL_DATASET_FILE, index=False)

metadata = {
    "low_variance_threshold": LOW_VARIANCE_THRESHOLD,
    "top_k_fscore": TOP_K_FSCORE,
    "final_rfe_features": len(final_features),
    "target_column": "bioactivity_class",
    "final_dataset": str(FINAL_DATASET_FILE),
}
(FS_DIR / "feature_selection_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")

print(f"Saved final feature-selected dataset: {FINAL_DATASET_FILE}")
final_dataset.head()

## Final outputs
- `final_selected_features_RFE.csv`
- `recA_RFE_final_modeling_dataset.csv`
- `03_rfe_feature_ranking.csv`